<h3>Initializing Spark Session</h3>

In [ ]:
#importing pyspark
import pyspark
#importing sparksession
from pyspark.sql import SparkSession
#creating a sparksession object and providing appName
spark=SparkSession.builder \
.master("local") \
.appName("CSCI-316 Project") \
.config('spark.ui.port','4050') \
.getOrCreate()

# Property used to format output tables better
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
#printing the version of spark
print("Apache Spark version: ", spark.version)
spark

Apache Spark version:  3.5.4


<h3>Loading the Dataset

In [ ]:
#get the dataset
!wget -q https://www.dubaipulse.gov.ae/dataset/c9263194-5ee3-4340-b7c0-3269b26acb43/resource/c3ece154-3071-4116-8650-e769d8416d88/download/traffic_incidents.csv

In [ ]:
df = spark.read.csv('traffic_incidents.csv', header=True, inferSchema=True)

<h3>Exploring Data</h3>

In [ ]:
df.show(truncate=False)

+----------+-------------------+-------------------------------------+-----------+-----------+
|acci_id   |acci_time          |acci_name                            |acci_x     |acci_y     |
+----------+-------------------+-------------------------------------+-----------+-----------+
|6209942016|06/02/2025 06:48:54|صدم عمود - بسيط                      |25.27055501|55.32972003|
|6209945908|06/02/2025 06:50:15|اصطدام بين مركبتين - بليغ            |25.05787   |55.13908999|
|6209973495|06/02/2025 07:00:05|تعطل مركبة ثقيلة - بسيط              |25.11327148|55.34880823|
|6210021477|06/02/2025 07:18:13|الوقوف خلف المركبات (دبل بارك) - بسيط|25.06172221|55.24508335|
|6210066063|06/02/2025 07:34:09|الوقوف خلف المركبات (دبل بارك) - بسيط|25.28251465|55.34662029|
|6210096611|06/02/2025 07:44:41|الوقوف خلف المركبات (دبل بارك) - بسيط|25.18973   |55.4282    |
|6210114782|06/02/2025 07:51:12|الوقوف خلف المركبات (دبل بارك) - بسيط|25.14690999|55.40375999|
|6210119242|06/02/2025 07:52:47|الوقوف خلف المركبا

In [ ]:
df.describe().show()

+-------+-------------------+-------------------+--------------------+-------------------+-------------------+
|summary|            acci_id|          acci_time|           acci_name|             acci_x|             acci_y|
+-------+-------------------+-------------------+--------------------+-------------------+-------------------+
|  count|              31586|              31586|               31586|              31586|              31586|
|   mean|5.912883109925125E9|               NULL|                NULL| 25.151265290927746|  55.29682691442739|
| stddev|5.367476959744801E8|               NULL|                NULL|0.17843691273130746|0.20292885380402562|
|    min|         4245262601|01/01/2025 00:04:02|ازدحام في المنافذ...|                0.0|        35.93452656|
|    max|         6210134214|31/12/2024 23:30:54|وجود جسم في الشار...|        31.94759659|        77.21667027|
+-------+-------------------+-------------------+--------------------+-------------------+-------------------+



In [ ]:
#print schema
df.printSchema()


root
 |-- acci_id: long (nullable = true)
 |-- acci_time: string (nullable = true)
 |-- acci_name: string (nullable = true)
 |-- acci_x: double (nullable = true)
 |-- acci_y: double (nullable = true)



<h3>Data Cleaning & Preprocessing</h3>

In [ ]:
#drop null values
df = df.dropna()

In [ ]:
#check for null values
from pyspark.sql.functions import *
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-------+---------+---------+------+------+
|acci_id|acci_time|acci_name|acci_x|acci_y|
+-------+---------+---------+------+------+
|      0|        0|        0|     0|     0|
+-------+---------+---------+------+------+



In [ ]:
#renaming columns
df = df.withColumnRenamed("acci_id", "Accident_ID") \
       .withColumnRenamed("acci_time", "Accident_Time") \
       .withColumnRenamed("acci_name", "Accident_Name") \
       .withColumnRenamed("acci_x", "Latitude") \
       .withColumnRenamed("acci_y", "Longitude")

In [ ]:
#convert the acci_time column to a timestamp datatype

df = df.withColumn("Accident_Time", to_timestamp("Accident_Time", "dd/MM/yyyy HH:mm:ss"))
df.show(truncate=False)


+-----------+-------------------+-------------------------------------+-----------+-----------+
|Accident_ID|Accident_Time      |Accident_Name                        |Latitude   |Longitude  |
+-----------+-------------------+-------------------------------------+-----------+-----------+
|6209942016 |2025-02-06 06:48:54|صدم عمود - بسيط                      |25.27055501|55.32972003|
|6209945908 |2025-02-06 06:50:15|اصطدام بين مركبتين - بليغ            |25.05787   |55.13908999|
|6209973495 |2025-02-06 07:00:05|تعطل مركبة ثقيلة - بسيط              |25.11327148|55.34880823|
|6210021477 |2025-02-06 07:18:13|الوقوف خلف المركبات (دبل بارك) - بسيط|25.06172221|55.24508335|
|6210066063 |2025-02-06 07:34:09|الوقوف خلف المركبات (دبل بارك) - بسيط|25.28251465|55.34662029|
|6210096611 |2025-02-06 07:44:41|الوقوف خلف المركبات (دبل بارك) - بسيط|25.18973   |55.4282    |
|6210114782 |2025-02-06 07:51:12|الوقوف خلف المركبات (دبل بارك) - بسيط|25.14690999|55.40375999|
|6210119242 |2025-02-06 07:52:47|الوقوف 

In [ ]:
#check the time datatype
df.printSchema()

root
 |-- Accident_ID: long (nullable = true)
 |-- Accident_Time: timestamp (nullable = true)
 |-- Accident_Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



In [ ]:
#Extract Hour & Weekday
df = df.withColumn("Hour", hour("Accident_Time"))
df = df.withColumn("Weekday", date_format("Accident_Time",'EEEE'))
df.show(truncate=False)


+-----------+-------------------+-------------------------------------+-----------+-----------+----+--------+
|Accident_ID|Accident_Time      |Accident_Name                        |Latitude   |Longitude  |Hour|Weekday |
+-----------+-------------------+-------------------------------------+-----------+-----------+----+--------+
|6209942016 |2025-02-06 06:48:54|صدم عمود - بسيط                      |25.27055501|55.32972003|6   |Thursday|
|6209945908 |2025-02-06 06:50:15|اصطدام بين مركبتين - بليغ            |25.05787   |55.13908999|6   |Thursday|
|6209973495 |2025-02-06 07:00:05|تعطل مركبة ثقيلة - بسيط              |25.11327148|55.34880823|7   |Thursday|
|6210021477 |2025-02-06 07:18:13|الوقوف خلف المركبات (دبل بارك) - بسيط|25.06172221|55.24508335|7   |Thursday|
|6210066063 |2025-02-06 07:34:09|الوقوف خلف المركبات (دبل بارك) - بسيط|25.28251465|55.34662029|7   |Thursday|
|6210096611 |2025-02-06 07:44:41|الوقوف خلف المركبات (دبل بارك) - بسيط|25.18973   |55.4282    |7   |Thursday|
|621011478

In [ ]:
df.select("Accident_Time", "Hour", "Weekday").show(300, truncate=False)


+-------------------+----+---------+
|Accident_Time      |Hour|Weekday  |
+-------------------+----+---------+
|2025-02-06 06:48:54|6   |Thursday |
|2025-02-06 06:50:15|6   |Thursday |
|2025-02-06 07:00:05|7   |Thursday |
|2025-02-06 07:18:13|7   |Thursday |
|2025-02-06 07:34:09|7   |Thursday |
|2025-02-06 07:44:41|7   |Thursday |
|2025-02-06 07:51:12|7   |Thursday |
|2025-02-06 07:52:47|7   |Thursday |
|2025-02-06 07:53:41|7   |Thursday |
|2025-02-06 07:57:49|7   |Thursday |
|2025-02-06 06:07:24|6   |Thursday |
|2025-02-06 06:21:04|6   |Thursday |
|2025-02-06 06:18:33|6   |Thursday |
|2025-02-06 06:25:36|6   |Thursday |
|2025-02-06 06:41:02|6   |Thursday |
|2025-02-06 06:49:31|6   |Thursday |
|2025-02-06 07:08:45|7   |Thursday |
|2025-02-06 07:14:46|7   |Thursday |
|2025-02-06 07:14:59|7   |Thursday |
|2025-02-06 07:21:29|7   |Thursday |
|2025-02-06 07:23:53|7   |Thursday |
|2025-02-06 07:28:20|7   |Thursday |
|2025-02-06 06:08:55|6   |Thursday |
|2025-02-06 06:07:39|6   |Thursday |
|

In [ ]:
#Extract Severity and Incident Type

# Extract the main incident type (before "-")
df = df.withColumn("Incident_Type", regexp_extract(df["Accident_Name"], r"^(.*?)\s-\s", 1))

# Extract Severity (after "-") and map to numeric values
df = df.withColumn("Severity", regexp_extract(df["Accident_Name"], r"-\s(.*)", 1))

# Convert Severity to numeric (بسيط = 0, بليغ = 1)
df = df.withColumn("Severity",
                   when(df["Severity"] == "بسيط", 0)
                   .when(df["Severity"] == "بليغ", 1)
                   .otherwise(None))

In [ ]:
df.show(truncate=False)


+-----------+-------------------+-------------------------------------+-----------+-----------+----+--------+------------------------------+--------+
|Accident_ID|Accident_Time      |Accident_Name                        |Latitude   |Longitude  |Hour|Weekday |Incident_Type                 |Severity|
+-----------+-------------------+-------------------------------------+-----------+-----------+----+--------+------------------------------+--------+
|6209942016 |2025-02-06 06:48:54|صدم عمود - بسيط                      |25.27055501|55.32972003|6   |Thursday|صدم عمود                      |0       |
|6209945908 |2025-02-06 06:50:15|اصطدام بين مركبتين - بليغ            |25.05787   |55.13908999|6   |Thursday|اصطدام بين مركبتين            |1       |
|6209973495 |2025-02-06 07:00:05|تعطل مركبة ثقيلة - بسيط              |25.11327148|55.34880823|7   |Thursday|تعطل مركبة ثقيلة              |0       |
|6210021477 |2025-02-06 07:18:13|الوقوف خلف المركبات (دبل بارك) - بسيط|25.06172221|55.24508335|7   |

In [ ]:
df.printSchema()

root
 |-- Accident_ID: long (nullable = true)
 |-- Accident_Time: timestamp (nullable = true)
 |-- Accident_Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Weekday: string (nullable = true)
 |-- Incident_Type: string (nullable = true)
 |-- Severity: integer (nullable = true)



In [ ]:

df = df.withColumn("Weekday_Num", dayofweek(col("Accident_Time")))
df.show(truncate=False)


+-----------+-------------------+-------------------------------------+-----------+-----------+----+--------+------------------------------+--------+-----------+
|Accident_ID|Accident_Time      |Accident_Name                        |Latitude   |Longitude  |Hour|Weekday |Incident_Type                 |Severity|Weekday_Num|
+-----------+-------------------+-------------------------------------+-----------+-----------+----+--------+------------------------------+--------+-----------+
|6209942016 |2025-02-06 06:48:54|صدم عمود - بسيط                      |25.27055501|55.32972003|6   |Thursday|صدم عمود                      |0       |5          |
|6209945908 |2025-02-06 06:50:15|اصطدام بين مركبتين - بليغ            |25.05787   |55.13908999|6   |Thursday|اصطدام بين مركبتين            |1       |5          |
|6209973495 |2025-02-06 07:00:05|تعطل مركبة ثقيلة - بسيط              |25.11327148|55.34880823|7   |Thursday|تعطل مركبة ثقيلة              |0       |5          |
|6210021477 |2025-02-06 07:1

In [ ]:
numerical_cols = ["Hour", "Weekday_num", "Latitude", "Longitude", "Severity"]

for col1 in numerical_cols:
    for col2 in numerical_cols:
        if col1 != col2:
            try:
                corr_value = df.stat.corr(col1, col2)
                print(f"Correlation between {col1} and {col2}: {corr_value:.3f}")
            except Exception as e:
                print(f"Error in {col1} vs {col2}: {e}")


Correlation between Hour and Weekday_num: -0.004
Correlation between Hour and Latitude: 0.002
Correlation between Hour and Longitude: 0.011
Correlation between Hour and Severity: 0.028
Correlation between Weekday_num and Hour: -0.004
Correlation between Weekday_num and Latitude: -0.012
Correlation between Weekday_num and Longitude: -0.012
Correlation between Weekday_num and Severity: -0.005
Correlation between Latitude and Hour: 0.002
Correlation between Latitude and Weekday_num: -0.012
Correlation between Latitude and Longitude: 0.173
Correlation between Latitude and Severity: -0.004
Correlation between Longitude and Hour: 0.011
Correlation between Longitude and Weekday_num: -0.012
Correlation between Longitude and Latitude: 0.173
Correlation between Longitude and Severity: -0.003
Correlation between Severity and Hour: 0.028
Correlation between Severity and Weekday_num: -0.005
Correlation between Severity and Latitude: -0.004
Correlation between Severity and Longitude: -0.003


In [ ]:
df_cleaned = df.select("Hour", "Weekday_Num","Accident_Time", "Latitude", "Longitude","Severity")
df_cleaned.show(truncate=False)

+----+-----------+-------------------+-----------+-----------+--------+
|Hour|Weekday_Num|Accident_Time      |Latitude   |Longitude  |Severity|
+----+-----------+-------------------+-----------+-----------+--------+
|6   |5          |2025-02-06 06:48:54|25.27055501|55.32972003|0       |
|6   |5          |2025-02-06 06:50:15|25.05787   |55.13908999|1       |
|7   |5          |2025-02-06 07:00:05|25.11327148|55.34880823|0       |
|7   |5          |2025-02-06 07:18:13|25.06172221|55.24508335|0       |
|7   |5          |2025-02-06 07:34:09|25.28251465|55.34662029|0       |
|7   |5          |2025-02-06 07:44:41|25.18973   |55.4282    |0       |
|7   |5          |2025-02-06 07:51:12|25.14690999|55.40375999|0       |
|7   |5          |2025-02-06 07:52:47|25.2926971 |55.38666495|0       |
|7   |5          |2025-02-06 07:53:41|25.23432691|55.29057357|0       |
|7   |5          |2025-02-06 07:57:49|25.25033999|55.30139999|0       |
|6   |5          |2025-02-06 06:07:24|24.94653556|55.1034521 |1 

In [ ]:
#check for null values
from pyspark.sql.functions import *
df_cleaned.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_cleaned.columns]).show()

+----+-----------+-------------+--------+---------+--------+
|Hour|Weekday_Num|Accident_Time|Latitude|Longitude|Severity|
+----+-----------+-------------+--------+---------+--------+
|   0|          0|            0|       0|        0|    2243|
+----+-----------+-------------+--------+---------+--------+



In [ ]:
#drop all the null values in the severity column in df_cleaned

df_cleaned = df_cleaned.dropna(subset=["Severity"])
df_cleaned.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_cleaned.columns]).show()


+----+-----------+-------------+--------+---------+--------+
|Hour|Weekday_Num|Accident_Time|Latitude|Longitude|Severity|
+----+-----------+-------------+--------+---------+--------+
|   0|          0|            0|       0|        0|       0|
+----+-----------+-------------+--------+---------+--------+



In [ ]:
from pyspark.ml.feature import VectorAssembler

# Select the features for X
assembler = VectorAssembler(inputCols=["Hour", "Weekday_Num", "Latitude", "Longitude"], outputCol="features")
X = assembler.transform(df_cleaned).select("features")

# Select the target variable for Y
Y = df_cleaned.select("Severity")

In [ ]:
from sklearn.preprocessing import StandardScaler
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, DoubleType
import numpy as np

to_array = udf(lambda v: v.toArray().tolist(), ArrayType(DoubleType()))

# Apply the UDF to the 'features' column
X = X.withColumn("features_array", to_array("features"))

# Convert to pandas DataFrame for compatibility with StandardScaler
X_pd = X.select("features_array").toPandas()

# Extract the NumPy array from the pandas DataFrame
X_np = X_pd['features_array'].apply(np.array).values

scaler = StandardScaler()
X = scaler.fit_transform(X_np.tolist())

## Ensemble Model

In [ ]:
import pandas as pd
X_scaled_pd = pd.DataFrame(X, columns=["Hour_scaled", "Weekday_Num_scaled", "Latitude_scaled", "Longitude_scaled"])
X_scaled_spark = spark.createDataFrame(X_scaled_pd)

In [ ]:
X_scaled_spark.show(truncate=False)

+-------------------+------------------+---------------------+---------------------+
|Hour_scaled        |Weekday_Num_scaled|Latitude_scaled      |Longitude_scaled     |
+-------------------+------------------+---------------------+---------------------+
|-1.1955178960350261|0.5016727062339991|0.6623838915562789   |0.164491969122588    |
|-1.1955178960350261|0.5016727062339991|-0.49745769042656574 |-0.7502307126628499  |
|-1.031058942122076 |0.5016727062339991|-0.19533512801658615 |0.25608513824988105  |
|-1.031058942122076 |0.5016727062339991|-0.47645031740799765 |-0.24163022917844545 |
|-1.031058942122076 |0.5016727062339991|0.7276037572042913   |0.24558648680758388  |
|-1.031058942122076 |0.5016727062339991|0.22161842594035225  |0.6370400641686339   |
|-1.031058942122076 |0.5016727062339991|-0.011893226521292908|0.5197666717376043   |
|-1.031058942122076 |0.5016727062339991|0.7831320190054578   |0.4377375181266067   |
|-1.031058942122076 |0.5016727062339991|0.46482010078309827  |-0.

In [ ]:
Y_scaled_pd = Y.toPandas()
Y_scaled_spark = spark.createDataFrame(Y_scaled_pd)

In [ ]:
# Assuming X_scaled_spark has an index column named 'index'
# and Y_scaled_spark also has an index column named 'index'

# Add an index column to both DataFrames if they don't already have one
from pyspark.sql.functions import monotonically_increasing_id

X_scaled_spark = X_scaled_spark.withColumn("index", monotonically_increasing_id())
Y_scaled_spark = Y_scaled_spark.withColumn("index", monotonically_increasing_id())

# Perform the join
combined = X_scaled_spark.join(Y_scaled_spark, on="index", how="inner").drop("index")

# Display the combined DataFrame
combined.show(truncate=False)

+-------------------+--------------------+---------------------+--------------------+--------+
|Hour_scaled        |Weekday_Num_scaled  |Latitude_scaled      |Longitude_scaled    |Severity|
+-------------------+--------------------+---------------------+--------------------+--------+
|-1.1955178960350261|0.5016727062339991  |0.18257252738166768  |-0.04663863016526728|0       |
|-1.5244358038609267|0.5016727062339991  |0.12094989644774005  |0.5439986843995145  |0       |
|0.2846126891815262 |-0.5569431090207008 |-0.8328375504851616  |-0.7131868750252431 |0       |
|-2.0178126655997777|-1.0862510166480508 |0.5012406025974857   |0.3525127618478335  |0       |
|0.4490716430944765 |1.030980613861349   |-0.03524564058007307 |-0.19760685213147033|0       |
|0.12015373526857595|1.030980613861349   |-0.7569873447346425  |-0.19270262865797333|0       |
|-1.031058942122076 |1.030980613861349   |0.41274668724526553  |-0.18131674966299724|0       |
|0.6135305970074267 |0.5016727062339991  |-1.21346

In [ ]:
# Convert the combined Spark DataFrame to a Pandas DataFrame
combined_pd = combined.toPandas()

# Inspect the first few rows
print(combined_pd.head())


   Hour_scaled  Weekday_Num_scaled  Latitude_scaled  Longitude_scaled  \
0    -1.195518            0.501673         0.182573         -0.046639   
1    -1.524436            0.501673         0.120950          0.543999   
2     0.284613           -0.556943        -0.832838         -0.713187   
3    -2.017813           -1.086251         0.501241          0.352513   
4     0.449072            1.030981        -0.035246         -0.197607   

   Severity  
0         0  
1         0  
2         0  
3         0  
4         0  


<h3>Model Building & Evaluation</h3>

In [ ]:
import numpy as np

# Define features and target
X = combined_pd[['Hour_scaled', 'Weekday_Num_scaled', 'Latitude_scaled', 'Longitude_scaled']].values
y = combined_pd['Severity'].values

# (Optional) Check the shapes
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (29343, 4)
y shape: (29343,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Assuming 'X' contains your features and 'y' contains your target variable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Now you can use X_train_res and y_train_res for training your model

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the SVM classifier
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)

# Train the SVM model
svm_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test)

# Evaluate the SVM model
print("----- SVM Model Evaluation -----")
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("SVM Classification Report:\n", classification_report(y_test, y_pred_svm))
print("SVM Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))


----- SVM Model Evaluation -----
SVM Accuracy: 0.5273470778667575
SVM Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.52      0.67      5268
           1       0.12      0.56      0.19       601

    accuracy                           0.53      5869
   macro avg       0.51      0.54      0.43      5869
weighted avg       0.83      0.53      0.62      5869

SVM Confusion Matrix:
 [[2761 2507]
 [ 267  334]]


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the k-NN classifier (using 5 neighbors as default)
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the k-NN model
knn_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
y_pred_knn = knn_model.predict(X_test)

# Evaluate the k-NN model
print("\n----- k-NN Model Evaluation -----")
print("k-NN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("k-NN Classification Report:\n", classification_report(y_test, y_pred_knn))
print("k-NN Confusion Matrix:\n", confusion_matrix(y_test, y_pred_knn))



----- k-NN Model Evaluation -----
k-NN Accuracy: 0.66280456636565
k-NN Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.70      0.79      5268
           1       0.12      0.35      0.18       601

    accuracy                           0.66      5869
   macro avg       0.51      0.53      0.48      5869
weighted avg       0.82      0.66      0.73      5869

k-NN Confusion Matrix:
 [[3678 1590]
 [ 389  212]]
